# PQID Gradio Space Audit and Deployment Notebook

This notebook is the auditable control surface for the PQID Hugging Face Gradio gateway. Run it before updating the Space, after changing dashboard files, and whenever you need to prove that the public dashboard still points to the intended dataset, GitHub snapshot, Zenodo DOI, figure assets, and release counts.

The notebook is intentionally written as a reusable template: update the configuration cell for a future dataset gateway, keep the structural checks, and adapt the expected files/counts.

## 1. Configure the audit

Edit only this cell when reusing the notebook for another project. The rest of the notebook derives paths and checks from these values.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import pandas as pd

SPACE_REPO_ID = "Elias-Abebe-Gasparini/PQID-Dataset-Gateway"
DATASET_REPO_ID = "Elias-Abebe-Gasparini/PQID"
EXPECTED_HF_DATASET_URL = "https://huggingface.co/datasets/Elias-Abebe-Gasparini/PQID"
EXPECTED_GITHUB_URL = "https://github.com/Elias-Abebe-Gasparini/PQID-Dataset/tree/v1.0.0-scientific-data-submission"
EXPECTED_ZENODO_URL = "https://doi.org/10.5281/zenodo.20024477"

EXPECTED_PUBLIC_OPEN_TOTAL = 311_724
EXPECTED_LICENSE_VALID_TOTAL = 319_782
EXPECTED_RESTRICTED_TOTAL = 230_532

EXPECTED_RUNTIME_FILES = ["app.py", "README.md", "requirements.txt"]
EXPECTED_AUDIT_FILES = [
    "check_gradio_space.py",
    "gradio_space_audit.ipynb",
    "run_local_space.ps1",
    "LOCAL_ENVIRONMENT_NOTE.md",
    "SPACE_UPLOAD_CHECKLIST.md",
]
EXPECTED_FIGURES = [
    "fig1_pqid_construction_pipeline_designed.png",
    "fig2_release_stratification_designed.png",
    "fig3_seed_generation_workflow_designed.png",
    "fig4_validation_audit_layers_designed.png",
    "fig5_readiness_statistics.png",
    "fig6_semantic_paraphrase_quality.png",
    "fig7_release_composition.png",
    "suppfig_s4_acquisition_pareto_diminishing_returns.png",
    "suppfig_s5_linguistic_distribution.png",
    "suppfig_s6_license_behavior_panel.png",
]

cwd = Path.cwd().resolve()
if (cwd / "app.py").exists() and (cwd / "README.md").exists():
    SPACE_DIR = cwd
elif (cwd / "PQID" / "platforms" / "gradio_space").exists():
    SPACE_DIR = cwd / "PQID" / "platforms" / "gradio_space"
else:
    raise FileNotFoundError("Could not locate PQID/platforms/gradio_space from the current working directory.")

FIGURE_DIR = SPACE_DIR / "assets" / "figures"
print("Space directory:", SPACE_DIR)
print("Notebook kernel:", sys.executable)
print("Python:", sys.version.split()[0])
SPACE_DIR

## 2. Inventory the Space package

This confirms which local files exist before upload. Runtime files are the only files required by Hugging Face Spaces; audit files stay local unless you intentionally upload them with `--include-audit-files`.

In [ ]:
inventory = []
for name in EXPECTED_RUNTIME_FILES + EXPECTED_AUDIT_FILES:
    path = SPACE_DIR / name
    inventory.append({
        "file": name,
        "kind": "runtime" if name in EXPECTED_RUNTIME_FILES else "audit/helper",
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else None,
    })
inventory_df = pd.DataFrame(inventory)
display(inventory_df)
assert inventory_df.loc[inventory_df["kind"] == "runtime", "exists"].all(), "Missing required runtime file."


## 3. Validate Hugging Face Space metadata

The front matter controls how Hugging Face builds the Space. The checks are intentionally simple so the notebook does not need a YAML dependency.

In [ ]:
readme = (SPACE_DIR / "README.md").read_text(encoding="utf-8")
assert readme.startswith("---\n"), "README.md must begin with Hugging Face YAML front matter."
front_matter = readme.split("---", 2)[1]

metadata_checks = {
    "sdk: gradio": "sdk: gradio" in front_matter,
    "app_file: app.py": "app_file: app.py" in front_matter,
    "python_version": "python_version" in front_matter,
    "dataset dependency": DATASET_REPO_ID in front_matter,
    "short description length <= 60": all(
        not line.startswith("short_description:") or len(line.split(":", 1)[1].strip().strip('"')) <= 60
        for line in front_matter.splitlines()
    ),
}
display(pd.DataFrame([{"check": k, "passed": v} for k, v in metadata_checks.items()]))
assert all(metadata_checks.values()), "One or more README metadata checks failed."


## 4. Check public links and private-marker hygiene

This checks that the dashboard points to the public release objects and does not contain known private project markers.

In [ ]:
app_text = (SPACE_DIR / "app.py").read_text(encoding="utf-8")
combined_public_text = app_text + "\n" + readme
required_links = [EXPECTED_HF_DATASET_URL, EXPECTED_GITHUB_URL, EXPECTED_ZENODO_URL]
for link in required_links:
    assert link in combined_public_text, f"Missing public link: {link}"

private_markers = [
    "FUNDING_PATHS",
    "PUBLICATION_TARGETS",
    "ACM_TQC_BENCHMARK_PAPER_DRAFT",
    "NATURE_MACHINE_INTELLIGENCE_PAPER_DRAFT",
    "OPENAI_RESEARCHER_ACCESS_APPLICATION",
]
found_private_markers = [term for term in private_markers if term in combined_public_text]
assert not found_private_markers, f"Potential private markers found: {found_private_markers}"
print("Public links present and configured private markers absent.")


## 5. Audit packaged figures

The Space serves figure previews from `assets/figures`. These files must be present before upload; otherwise the figures tab will show missing-image behavior.

In [ ]:
figure_rows = []
for name in EXPECTED_FIGURES:
    path = FIGURE_DIR / name
    figure_rows.append({
        "figure_file": name,
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else None,
    })
figures_df = pd.DataFrame(figure_rows)
display(figures_df)
assert figures_df["exists"].all(), "One or more figure assets are missing."


## 6. Run the structural package checker

`check_gradio_space.py` is the command-line version of this audit. It is useful for CI, terminal checks, and quick pre-upload validation.

In [ ]:
checker = SPACE_DIR / "check_gradio_space.py"
assert checker.exists(), "Missing check_gradio_space.py"
result = subprocess.run(
    [sys.executable, str(checker), "--skip-import"],
    cwd=SPACE_DIR,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print("STDERR:\n", result.stderr)
assert result.returncode == 0, "Structural package checker failed."


## 7. Import the app and inspect release-integrity checks

This imports `app.py` without launching the server. It verifies that the dashboard exposes a Gradio `demo` object and that the release-integrity table can be constructed.

In [ ]:
missing_packages = [pkg for pkg in ["gradio", "datasets", "huggingface_hub", "pandas"] if importlib.util.find_spec(pkg) is None]
if missing_packages:
    raise ModuleNotFoundError(f"Install runtime packages first: {missing_packages}")

os.environ.setdefault("GRADIO_ANALYTICS_ENABLED", "False")
sys.path.insert(0, str(SPACE_DIR))
import app

assert hasattr(app, "demo"), "app.py does not expose a Gradio demo object."
integrity_df = app.release_integrity_table()
display(integrity_df)
assert set(integrity_df["status"]).issubset({"OK", "CHECK"})
assert (integrity_df["status"] == "OK").all(), "One or more static release-integrity checks require attention."
type(app.demo).__name__


## 8. Optional live Hugging Face summary check

Set `RUN_LIVE_HF_CHECK = True` only when you have network access. This checks the public summary JSON currently hosted on the Hugging Face dataset repository against the expected public-open total.

In [ ]:
RUN_LIVE_HF_CHECK = False

if RUN_LIVE_HF_CHECK:
    live_table, live_summary = app.run_live_release_integrity_check()
    display(live_table)
    display(live_summary)
    assert (live_table["status"] == "OK").all(), "Live integrity check requires attention."
else:
    print("Skipped live Hugging Face check. Set RUN_LIVE_HF_CHECK = True to run it.")


## 9. Optional local preview

Run this cell only when you want to inspect the dashboard locally. Stop the cell/kernel when finished. If your local environment is troublesome, skip this and rely on the Hugging Face Space rebuild.

In [ ]:
RUN_LOCAL_PREVIEW = False

if RUN_LOCAL_PREVIEW:
    app.demo.launch(server_name="127.0.0.1", server_port=7860, prevent_thread_lock=False)
else:
    print("Skipped local preview. Set RUN_LOCAL_PREVIEW = True to launch locally.")


## 10. Optional upload to Hugging Face Spaces

This guarded cell uses `upload_space.py`, the same script used for the live PQID gateway. It uploads runtime files and figure assets by default. Leave `RUN_UPLOAD = False` until the previous checks pass and you are ready to deploy.

In [ ]:
RUN_UPLOAD = False
INCLUDE_AUDIT_FILES_IN_SPACE = False

if RUN_UPLOAD:
    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
    upload_script = SPACE_DIR / "upload_space.py"
    command = [sys.executable, str(upload_script), "--repo-id", SPACE_REPO_ID]
    if INCLUDE_AUDIT_FILES_IN_SPACE:
        command.append("--include-audit-files")
    result = subprocess.run(command, cwd=SPACE_DIR, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print("STDERR:\n", result.stderr)
    assert result.returncode == 0, "Space upload failed."
else:
    print("Skipped upload. Set RUN_UPLOAD = True when ready to deploy.")


## 11. Reuse checklist for future projects

When adapting this notebook and dashboard template:

1. Replace the repo IDs, public URLs, expected totals, and expected figure filenames in the configuration cell.
2. Update `README.md` Hugging Face metadata and `requirements.txt`.
3. Keep runtime uploads minimal: `app.py`, `README.md`, `requirements.txt`, and public assets only.
4. Keep audit notebooks/scripts local unless they are intentionally public and scrubbed.
5. Run the structural checker, import check, figure audit, and release-integrity table before every Space upload.
6. After upload, rebuild/restart the Space and verify the live dashboard manually in both light and dark modes.